# LLM Evaluation

Companion notebook for the [LLM Evaluation lesson](https://ml-viz-ruby.vercel.app/courses/model-evaluation/04-llm-evaluation).

**The idea in one sentence.** Evaluating a language model is hard because there's no
single label — so we lean on **perplexity** (how surprised is the model?),
**bits-per-byte** (a tokenizer-fair version), **AI-as-a-judge** (with its biases),
and **Bradley–Terry/Elo** to turn pairwise preferences into a ranking.

What this notebook builds and validates:

- **Entropy → cross-entropy → perplexity**, with the irreducible floor $2^{H(q)}$
  reached exactly when the model matches the data.
- **Bits-per-byte** — perplexity is tokenizer-dependent; BPB is not.
- **Judge position bias** — a tied comparison isn't 50/50 if you fix the order.
- **Bradley–Terry MLE** — recover latent skills (Elo) from pairwise wins.

We **validate that perplexity bottoms out at the data floor, that BPB is
tokenizer-invariant, and that the BT fit recovers the true ranking**.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

BRAND  = '#6366f1'
TEAL   = '#2dd4bf'
ROSE   = '#fb7185'
ORANGE = '#f97316'
YELLOW = '#facc15'
MUTED  = '#475569'

## 1. Entropy, cross-entropy, perplexity on a toy vocabulary

We have a tiny vocabulary of 8 tokens. The **data distribution** $q$ is what the world actually emits at this position; the **model distribution** $p_\theta$ is what our model predicts. Cross-entropy and perplexity measure how far $p_\theta$ is from $q$.

$$ H(q, p_\theta) = -\sum_i q_i \log_2 p_{\theta,i} \qquad \mathrm{PPL} = 2^{H(q, p_\theta)} $$

Key limits to keep in your head: $H(q, q) = H(q)$ (the entropy of the data itself — the **floor**), and $H(q, p_\theta) \geq H(q)$ for any $p_\theta$ by Gibbs' inequality.

In [ ]:
# Data distribution q over 8 tokens (somewhat peaky, but not a delta)
V = 8
q = np.array([0.30, 0.22, 0.15, 0.12, 0.08, 0.06, 0.04, 0.03])
assert np.isclose(q.sum(), 1.0)

def entropy_bits(p):
    p = np.clip(p, 1e-12, 1.0)
    return -float((p * np.log2(p)).sum())

def cross_entropy_bits(q, p):
    p = np.clip(p, 1e-12, 1.0)
    return -float((q * np.log2(p)).sum())

def perplexity(q, p):
    return 2.0 ** cross_entropy_bits(q, p)

H_q = entropy_bits(q)
print(f'Floor: H(q) = {H_q:.3f} bits, PPL_floor = 2^H(q) = {2**H_q:.3f}')
print(f'Uniform-model CE   = {cross_entropy_bits(q, np.ones(V)/V):.3f}, PPL = {perplexity(q, np.ones(V)/V):.3f}  (= V = {V})')
print(f'Self CE H(q, q)    = {cross_entropy_bits(q, q):.3f}, PPL = {perplexity(q, q):.3f}  (matches the floor)')

A perfect model that matched the data exactly would land at the floor PPL of $\approx 5.7$ — the effective branching factor of $q$ itself. A uniform model spends $\log_2 8 = 3$ bits per token and has PPL $= 8$ (it cannot beat "guess uniformly from the vocabulary"). Now let's see how PPL responds to a **temperature mismatch** between the model and the data.

In [ ]:
# Temperature-scaled copies of q, then renormalise.
# T=1 -> matches q; T<1 -> peakier than q; T>1 -> flatter than q.
def temper(q, T, eps=1e-12):
    logits = np.log(np.clip(q, eps, 1.0))
    z = logits / T
    z = z - z.max()
    p = np.exp(z)
    return p / p.sum()

Ts = np.linspace(0.3, 3.0, 28)
ppls = [perplexity(q, temper(q, T)) for T in Ts]

fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.plot(Ts, ppls, color=BRAND, linewidth=2, label='PPL(data, temper(q, T))')
ax.axhline(2**H_q, color=TEAL,  linestyle='--', linewidth=1, label=f'Floor: PPL = {2**H_q:.2f}')
ax.axhline(V,      color=ROSE,  linestyle='--', linewidth=1, label=f'Uniform: PPL = {V}')
ax.axvline(1.0,    color=MUTED, linestyle=':',  linewidth=1)
ax.set_xlabel('temperature T applied to model copy of q')
ax.set_ylabel('perplexity')
ax.set_title('Perplexity vs temperature mismatch')
ax.legend(loc='upper left', frameon=False)
ax.grid(True)
plt.tight_layout(); plt.show()

The curve touches the floor at exactly $T = 1$ (the model matches the data) and grows in both directions: $T < 1$ overcommits to high-probability tokens, $T > 1$ collapses toward uniform. Both increase cross-entropy, and PPL grows multiplicatively. This is the same shape you would see plotting validation perplexity vs a *miscalibrated* model.

Now let's simulate **convergence**: a model that starts at the uniform distribution and is gradually pulled toward $q$. We expect PPL to drop monotonically from $V = 8$ toward the floor.

In [ ]:
uniform = np.ones(V) / V
alphas = np.linspace(0.0, 1.0, 41)            # interpolate model = (1-a)*uniform + a*q
ppls_conv = [perplexity(q, (1 - a) * uniform + a * q) for a in alphas]

fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.plot(alphas, ppls_conv, color=BRAND, linewidth=2, label='PPL as model converges to q')
ax.axhline(2**H_q, color=TEAL, linestyle='--', linewidth=1, label=f'Floor: PPL = {2**H_q:.2f}')
ax.set_xlabel(r'mixing weight $\alpha$ (model = (1-$\alpha$) uniform + $\alpha$ q)')
ax.set_ylabel('perplexity')
ax.set_title('Model converging from uniform to data')
ax.legend(loc='upper right', frameon=False)
ax.grid(True)
plt.tight_layout(); plt.show()

print(f'PPL at alpha=0.0  (uniform): {ppls_conv[0]:.3f}')
print(f'PPL at alpha=0.5  (halfway): {ppls_conv[20]:.3f}')
print(f'PPL at alpha=1.0  (perfect): {ppls_conv[-1]:.3f}')

### Validate: perplexity bottoms out at the data floor when the model matches

Cross-entropy $H(q,p)\ge H(q)$ with equality iff $p=q$ (Gibbs' inequality), so
perplexity $2^{H(q,p)}$ is minimised — at exactly the floor $2^{H(q)}$ — when the
model equals the data distribution. We confirm the temperature sweep bottoms out at
$T=1$ (where `temper(q,1)=q`) and hits the floor.

In [ ]:
floor = 2 ** H_q
ppl_at_1 = perplexity(q, temper(q, 1.0))
print(f'perplexity floor 2^H(q) = {floor:.4f}')
print(f'perplexity at T=1       = {ppl_at_1:.4f}  (model == data)')
assert np.isclose(ppl_at_1, floor), 'PPL at T=1 must equal the entropy floor'
assert min(ppls) >= floor - 1e-9, 'no model can beat the entropy floor'
best_T = Ts[int(np.argmin(ppls))]
print(f'temperature minimising PPL over the sweep: T = {best_T:.2f}  (~1.0)')
assert abs(best_T - 1.0) < 0.15, 'perplexity is minimised where the model matches the data'
print('\n✅ perplexity is minimised at the data distribution and can never beat 2^H(q)')

## 2. Bits-per-byte: a tokenizer-invariant metric

Perplexity divides total information by **token count**. If two models use different tokenizers — fine BPE vs coarse SentencePiece — the same text yields different $N$, and PPL is no longer comparable. **Bits-per-byte** normalises by the **byte count** of the raw text:

$$ \mathrm{BPB} \;=\; \frac{-\sum_i \log_2 p_\theta(x_i)}{B}, $$

where $B$ is the total bytes. Bytes are tokenizer-invariant, so BPB is comparable across architectures and vocab sizes. We'll demonstrate this on a tiny synthetic corpus: the *same* underlying text encoded under two simulated tokenizers, where each produces the same total bit-budget for the corpus but very different per-token counts.

In [ ]:
# Same raw text, two simulated tokenizers.
# We pretend the corpus is 1000 bytes of English text. Two tokenizers chop it
# into different numbers of tokens, but the *total information* in the text
# (the sum of -log p of each token's probability) is comparable because both
# tokenizers are decoding the same byte stream.

B_BYTES = 1000

# Tokenizer A: fine-grained BPE. ~4 bytes/token -> 250 tokens.
# Per-token log-probs sampled to give an average per-token surprisal of 4 bits.
rng = np.random.default_rng(7)
N_A = 250
bits_per_token_A = rng.normal(loc=4.0, scale=0.6, size=N_A).clip(0.5, None)
total_bits_A = bits_per_token_A.sum()

# Tokenizer B: coarse SentencePiece. ~8 bytes/token -> 125 tokens.
# Same underlying text -> same total bits, but spread over fewer tokens, so
# per-token surprisal is ~8 bits.
N_B = 125
bits_per_token_B = rng.normal(loc=8.0, scale=0.9, size=N_B).clip(0.5, None)
# Renormalise B's total to match A's so the *underlying* model is the same on
# both tokenisations (the only thing different is how we count tokens).
bits_per_token_B *= total_bits_A / bits_per_token_B.sum()
total_bits_B = bits_per_token_B.sum()

print(f'Tokenizer A: N = {N_A} tokens, total bits = {total_bits_A:.1f}, bytes = {B_BYTES}')
print(f'Tokenizer B: N = {N_B} tokens, total bits = {total_bits_B:.1f}, bytes = {B_BYTES}')

# Per-token cross-entropy and perplexity
H_A   = total_bits_A / N_A
H_B   = total_bits_B / N_B
PPL_A = 2 ** H_A
PPL_B = 2 ** H_B

# Bits-per-byte (tokenizer-invariant)
BPB_A = total_bits_A / B_BYTES
BPB_B = total_bits_B / B_BYTES

print(f'\nPerplexity  Tokenizer A: {PPL_A:7.2f}')
print(f'Perplexity  Tokenizer B: {PPL_B:7.2f}    ← huge gap, but same underlying text!')
print(f'\nBits-per-byte A: {BPB_A:.3f}')
print(f'Bits-per-byte B: {BPB_B:.3f}    ← identical, as it should be')

Per-token perplexity disagrees wildly between the two tokenizers (tens vs hundreds) even though the underlying model is encoding the same byte stream with the same total information. **Bits-per-byte agrees to within rounding** because it normalises by the byte count, which is fixed by the text, not by the tokenizer. This is exactly why papers like GPT-3, Chinchilla, LLaMA, and Pythia all report BPB on shared corpora — it is the only fair cross-tokenizer metric.

In [ ]:
# Bar chart: PPL diverges, BPB agrees
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 3.3))
ax1.bar(['Tokenizer A\n(fine BPE)', 'Tokenizer B\n(coarse SP)'], [PPL_A, PPL_B], color=[BRAND, ROSE])
ax1.set_title('Perplexity (tokenizer-dependent)')
ax1.set_ylabel('PPL')
ax1.grid(True, axis='y')
ax2.bar(['Tokenizer A\n(fine BPE)', 'Tokenizer B\n(coarse SP)'], [BPB_A, BPB_B], color=[BRAND, ROSE])
ax2.set_title('Bits-per-byte (tokenizer-invariant)')
ax2.set_ylabel('BPB')
ax2.grid(True, axis='y')
plt.tight_layout(); plt.show()

### Validate: bits-per-byte is tokenizer-invariant, perplexity is not

Per-token perplexity depends on how the text was chopped — a coarse tokenizer packs
more surprisal per token, inflating PPL — even for the *same* underlying model on the
*same* bytes. Bits-per-byte divides total bits by the byte count, cancelling the
tokenization, so it agrees across tokenizers. That's why cross-model LM comparisons
report BPB.

In [ ]:
print(f'perplexity : A = {PPL_A:.2f}, B = {PPL_B:.2f}  (differ by {abs(PPL_A-PPL_B):.1f} -> tokenizer-dependent)')
print(f'bits/byte  : A = {BPB_A:.4f}, B = {BPB_B:.4f}  (nearly equal -> tokenizer-invariant)')
assert abs(PPL_A - PPL_B) > 1.0, 'per-token perplexity should disagree across tokenizers'
assert abs(BPB_A - BPB_B) < 0.05 * max(BPB_A, BPB_B), 'BPB should agree across tokenizers'
print('\n✅ compare models with bits-per-byte, not perplexity, when tokenizers differ')

## 3. AI-as-a-judge position bias

We simulate a judge LLM scoring head-to-head pairs of answers from two **genuinely-tied** models, $X$ and $Y$. Without position bias the judge would call $X$'s win-rate $\approx 50\%$. We give the judge a 65/35 bias toward whichever answer is shown **first** — a magnitude in the range reported for production judge models. Then we compare two protocols:

- **Fixed order**: $X$ is always slot A (first). The bias acts as a constant skew toward $X$.
- **Randomised order**: a coin flip per trial decides whether $X$ is slot A or slot B. The bias is the same magnitude but its direction flips, so on average it cancels.

The headline metric is $X$'s reported win-rate over $1000$ trials. The randomised protocol should converge to $\approx 50\%$; the fixed protocol should converge to $\approx 65\%$.

In [ ]:
POS_BIAS = 0.65    # P(judge picks slot A | tied content)
N_TRIALS = 1000

def simulate(fixed_order: bool, n_trials: int, seed: int = 0):
    """Return X's reported win-rate under the given protocol.

    The two underlying models are genuinely tied. The judge has a POS_BIAS
    probability of picking whichever answer is in slot A.
    """
    rng = np.random.default_rng(seed)
    x_wins = 0
    for _ in range(n_trials):
        # Decide which slot X occupies
        x_in_A = True if fixed_order else bool(rng.integers(0, 2))
        # Judge picks slot A with prob POS_BIAS, slot B otherwise
        picked_A = rng.random() < POS_BIAS
        # Did X win this trial?
        if (x_in_A and picked_A) or ((not x_in_A) and (not picked_A)):
            x_wins += 1
    return x_wins / n_trials

win_fixed = simulate(fixed_order=True,  n_trials=N_TRIALS, seed=1)
win_rand  = simulate(fixed_order=False, n_trials=N_TRIALS, seed=2)
print(f'True (latent) win-rate of X: 50.0 %')
print(f'Reported win-rate, FIXED order  : {win_fixed*100:.1f} %  (skewed by position bias)')
print(f'Reported win-rate, RANDOMISED   : {win_rand*100:.1f} %  (bias cancels on average)')

In [ ]:
# Sweep over seeds to plot the distribution of reported win-rates.
seeds = range(60)
wins_fixed = [simulate(True,  N_TRIALS, seed=s) for s in seeds]
wins_rand  = [simulate(False, N_TRIALS, seed=s) for s in seeds]

fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(0.40, 0.75, 25)
ax.hist(wins_fixed, bins=bins, alpha=0.7, color=ROSE,  label=f'Fixed order (mean = {np.mean(wins_fixed):.3f})')
ax.hist(wins_rand,  bins=bins, alpha=0.7, color=TEAL,  label=f'Randomised order (mean = {np.mean(wins_rand):.3f})')
ax.axvline(0.50,       color=MUTED, linestyle='--', linewidth=1, label='True 50/50')
ax.axvline(POS_BIAS,   color=ROSE,  linestyle=':',  linewidth=1, label=f'Expected fixed-order skew = {POS_BIAS:.2f}')
ax.set_xlabel("Reported win-rate of X (over 1000 trials)")
ax.set_ylabel('# of seeds')
ax.set_title('AI-as-a-judge position bias: fixed vs randomised slot')
ax.legend(loc='upper left', frameon=False, fontsize=9)
ax.grid(True)
plt.tight_layout(); plt.show()

The fixed-order distribution sits tightly around $0.65$ — a 15-point systematic skew that no amount of additional data will remove, because the bias is structural. The randomised-order distribution sits around $0.50$ with normal-looking noise. **Randomising the A/B slot per trial is the single most effective protocol fix.** Production judge harnesses (MT-Bench, AlpacaEval) and the LMSYS Chatbot Arena pipeline all do this; the rest of the bias toolkit (chain-of-thought judging, multi-judge ensembles, human calibration sets) only matters once order is randomised.

## 4. Bradley–Terry / Elo from pairwise wins

Given a matrix of pairwise wins between $K$ models, we want to recover one **latent skill** $\theta_i$ per model such that

$$ P(i \text{ beats } j) = \sigma(\theta_i - \theta_j) = \frac{1}{1 + e^{-(\theta_i - \theta_j)}}. $$

We fit $\theta$ by maximising the log-likelihood of the observed wins. This is convex, so plain gradient ascent on $\theta$ converges to the MLE. The conversion to **Elo points** is the conventional linear rescale $\theta \mapsto 400/\ln 10 \cdot \theta$ (so a 1-natural-log advantage equals $\approx 173.7$ Elo points, the same constant used by FIDE and Chatbot Arena).

In [ ]:
# True latent skills for K=4 simulated models (in natural-log Elo units).
# Big spread on purpose so the MLE is well-identified.
true_theta = np.array([0.0, 0.4, -0.6, 0.9])
K = len(true_theta)
MODELS = ['M-A', 'M-B', 'M-C', 'M-D']

# Simulate N_PAIRS random pairings; each pairing is decided by the BT model.
N_PAIRS = 4000
rng = np.random.default_rng(11)
pair_a = rng.integers(0, K, size=N_PAIRS)
pair_b = rng.integers(0, K, size=N_PAIRS)
same = pair_a == pair_b
pair_b = np.where(same, (pair_b + 1) % K, pair_b)        # disallow self-matches
p_win_a = 1.0 / (1.0 + np.exp(-(true_theta[pair_a] - true_theta[pair_b])))
wins_a  = (rng.random(N_PAIRS) < p_win_a).astype(int)    # 1 if A won, else B won

# Aggregate into a K x K win-count matrix W where W[i, j] = # times i beat j.
W = np.zeros((K, K), dtype=int)
for a, b, w in zip(pair_a, pair_b, wins_a):
    if w == 1:
        W[a, b] += 1
    else:
        W[b, a] += 1

print('Win-count matrix W[i, j] = # times row i beat col j:')
print(W)

In [ ]:
def fit_bradley_terry(W, lr=0.5, n_steps=2000, eps=1e-9):
    """Maximum-likelihood fit of Bradley–Terry skills theta from a K x K win matrix.

    log L = sum_{i, j} W[i, j] * log sigmoid(theta_i - theta_j)
    grad_i = sum_j (W[i, j] - (W[i, j] + W[j, i]) * sigmoid(theta_i - theta_j))
    Identifiability: anchor mean(theta) = 0 after each step.
    """
    K = W.shape[0]
    theta = np.zeros(K)
    history = []
    for step in range(n_steps):
        diff   = theta[:, None] - theta[None, :]                # (K, K)
        sig    = 1.0 / (1.0 + np.exp(-diff))
        N_pair = W + W.T
        grad   = (W - N_pair * sig).sum(axis=1) / max(N_pair.sum(), 1)  # normalise by #games
        theta += lr * grad
        theta -= theta.mean()                                    # anchor
        if step % 100 == 0:
            logL = float((W * np.log(np.clip(sig, eps, 1.0))).sum())
            history.append(logL)
    return theta, history

theta_hat, ll_history = fit_bradley_terry(W)

ELO_K = 400.0 / np.log(10.0)
true_elo = (true_theta - true_theta.mean()) * ELO_K
rec_elo  = theta_hat * ELO_K

print('Model | true Elo (anchored to mean=0) | recovered Elo')
print('-' * 56)
for name, t, r in zip(MODELS, true_elo, rec_elo):
    print(f'{name:5s} |        {t:+8.1f}            |   {r:+8.1f}')

In [ ]:
# Visualise: log-likelihood climbing, then recovered vs true Elo
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.6))

ax1.plot(np.arange(len(ll_history)) * 100, ll_history, color=BRAND, linewidth=2)
ax1.set_xlabel('gradient step')
ax1.set_ylabel('log-likelihood')
ax1.set_title('Bradley–Terry MLE converging')
ax1.grid(True)

ax2.scatter(true_elo, rec_elo, color=BRAND, s=110, edgecolor='#0f1117', zorder=3)
lo, hi = min(true_elo.min(), rec_elo.min()) - 30, max(true_elo.max(), rec_elo.max()) + 30
ax2.plot([lo, hi], [lo, hi], color=MUTED, linestyle='--', linewidth=1, label='y = x')
for name, t, r in zip(MODELS, true_elo, rec_elo):
    ax2.annotate(name, (t, r), textcoords='offset points', xytext=(7, 6), color='#e2e8f0')
ax2.set_xlabel('true Elo')
ax2.set_ylabel('recovered Elo')
ax2.set_xlim(lo, hi); ax2.set_ylim(lo, hi)
ax2.set_title('Recovered ratings track ground truth')
ax2.legend(frameon=False)
ax2.grid(True)
plt.tight_layout(); plt.show()

### Validate: the Bradley–Terry fit recovers the true ranking

Bradley–Terry models $P(i \text{ beats } j)=\sigma(\theta_i-\theta_j)$; fitting
$\theta$ by MLE from a win matrix should recover the *ordering* of the true latent
skills (and their spacings up to a constant, which we anchor at mean 0). We check the
recovered order matches the truth.

In [ ]:
order_true = list(np.argsort(true_theta))
order_rec  = list(np.argsort(rec_elo))
print('true skill order   :', [MODELS[i] for i in order_true])
print('recovered order    :', [MODELS[i] for i in order_rec])
assert order_true == order_rec, 'BT MLE should recover the true ranking'
# and the recovered Elo correlates strongly with the true Elo
corr = np.corrcoef(true_elo, rec_elo)[0, 1]
print(f'correlation(true Elo, recovered Elo): {corr:.3f}')
assert corr > 0.98, 'recovered Elo should track the true skills'
print('\n✅ Bradley–Terry recovers the ranking (and spacings) from pairwise wins alone')

The fit recovers the true ordering and the spacings between models, off by a small amount due to the finite number of pairwise comparisons. This is the same algorithm running under the Chatbot Arena hood (with a larger $K$, hundreds of thousands of pairs, and a few extra terms for ties and bootstrap CIs). It is also the same algorithm that fits an **RLHF reward model** from human pairwise preferences — the Bradley–Terry log-likelihood reappears as the reward-model training loss, with the reward head's outputs playing the role of $\theta$.

---
## ✏️ Your turn

### Exercise: implement `bits_per_byte(log2_probs, byte_count)`

Given a list/array `log2_probs` of $\log_2 p_\theta(x_i)$ for each token in a corpus (so each entry is **negative**), and the byte count `byte_count` of the underlying raw text, return the **bits-per-byte**

$$ \mathrm{BPB} \;=\; \frac{-\sum_i \log_2 p_\theta(x_i)}{B}. $$

The tests below check:
1. A hand-computed reference on a tiny 4-token example.
2. Invariance to splitting the *same* underlying text into a different number of tokens, as long as the total bits and byte count are unchanged.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **perplexity across tokenizers** | not comparable — a coarser tokenizer inflates PPL; use BPB |
| **judge position bias** | fixed answer order skews tied matchups (demo); randomise/swap |
| **judge self-preference & verbosity bias** | LLM judges favour their own style and longer answers |
| **Elo needs enough games** | sparse or non-transitive matchups make skills ill-identified |
| **benchmark contamination** | test items leaking into pretraining inflate scores |

Demo: randomising answer order removes an LLM judge's position bias.

In [ ]:
# Judge position bias is a real threat to leaderboards: with a fixed answer order,
# two TIED models produce a systematically skewed win-rate. Randomising the order
# (or averaging both orders) restores the honest 50/50 — the standard mitigation.
mean_fixed = np.mean(wins_fixed)
mean_rand  = np.mean(wins_rand)
print(f'reported win-rate for TIED models — fixed order: {mean_fixed:.3f}  (biased toward {POS_BIAS})')
print(f'reported win-rate for TIED models — randomised:  {mean_rand:.3f}  (~0.50, honest)')
assert abs(mean_fixed - 0.5) > abs(mean_rand - 0.5), 'randomising order reduces position bias'
print('\nAlways randomise or swap answer order when using an LLM judge, or the leaderboard lies.')

In [ ]:
def bits_per_byte(log2_probs, byte_count):
    '''Bits-per-byte for a sequence of token log-probabilities.

    Args:
        log2_probs: 1D array of log2 probabilities, one per token. Each entry
            is <= 0 (because probabilities are in (0, 1]).
        byte_count: number of bytes in the raw text those tokens decode to.

    Returns:
        BPB = (- sum of log2 probs) / byte_count.
    '''
    # TODO(you):
    #   1. Sum the log2_probs.
    #   2. Negate to get the total bits used to encode the corpus.
    #   3. Divide by byte_count.
    pass

# Smoke run
out = bits_per_byte(np.array([-1.0, -2.0, -2.0, -1.0]), byte_count=12)
print(f'sample BPB: {out}')

In [ ]:
# Test 1: hand-computed reference
#   log2_probs = [-1, -2, -2, -1]  ->  total bits = 1 + 2 + 2 + 1 = 6
#   byte_count = 12               ->  BPB = 6 / 12 = 0.5
bpb_ref = bits_per_byte(np.array([-1.0, -2.0, -2.0, -1.0]), byte_count=12)
assert bpb_ref is not None, 'bits_per_byte returned None'
assert np.isfinite(bpb_ref), f'BPB is not finite: {bpb_ref}'
assert abs(bpb_ref - 0.5) < 1e-9, f'expected BPB = 0.5, got {bpb_ref}'

# Test 2: tokenizer invariance.
# A finer tokenizer chops the same text into 8 tokens with smaller per-token surprisal,
# but the total bits and byte count are identical. BPB must agree.
bpb_fine = bits_per_byte(np.array([-0.75]*8), byte_count=12)   # total bits = 6 -> BPB = 0.5
bpb_coarse = bits_per_byte(np.array([-1.5]*4), byte_count=12)  # total bits = 6 -> BPB = 0.5
assert abs(bpb_fine - 0.5) < 1e-9, f'fine: expected 0.5, got {bpb_fine}'
assert abs(bpb_coarse - 0.5) < 1e-9, f'coarse: expected 0.5, got {bpb_coarse}'
assert abs(bpb_fine - bpb_coarse) < 1e-9, 'BPB must be invariant to tokenizer granularity'

# Test 3: BPB must be non-negative
assert bpb_ref >= 0, f'BPB should be non-negative, got {bpb_ref}'

print(f'hand-computed BPB: {bpb_ref:.4f}   (expected 0.5)')
print(f'fine tokenizer:    {bpb_fine:.4f}   (expected 0.5)')
print(f'coarse tokenizer:  {bpb_coarse:.4f}   (expected 0.5)')
print('✅ Exercise passed')

<details>
<summary>💡 Show solution</summary>

```python
def bits_per_byte(log2_probs, byte_count):
    log2_probs = np.asarray(log2_probs, dtype=float)
    total_bits = -float(log2_probs.sum())
    return total_bits / byte_count
```

Two things worth noticing:

1. **Sum the log-probs, then negate.** Equivalently, you could sum `-log2_probs` and skip the outer negate — same thing, just less surprising arithmetic.
2. **The byte count is fixed by the raw text.** That's the whole point of BPB: the numerator changes with the model's quality, and the denominator changes with the text's length but **not** with how the tokenizer chops it up. That is why BPB is the right cross-tokenizer comparison metric — PPL is not.
</details>

## Key takeaways

- **Perplexity = $2^{\text{cross-entropy}}$**, minimised at the data floor
  $2^{H(q)}$ when the model matches the data (verified) — it can never go lower.
- **Bits-per-byte is tokenizer-invariant; perplexity is not** — use BPB to compare
  models with different tokenizers (verified).
- **AI-judges have position bias:** a fixed answer order skews tied comparisons;
  randomise or swap the order (demo).
- **Bradley–Terry/Elo turns pairwise wins into a ranking** — the MLE recovered the
  true order and spacings from wins alone.